# 🔧 Tool Use Basics

## Use case

Les LLMs sont limités à leur connaissance d'entraînement. Les **tools** (ou function calling) permettent de connecter le modèle à des systèmes externes : APIs, bases de données, moteurs de recherche.

Le modèle ne *execute* pas les tools lui-même, il décide *lesquels appeler* et *avec quels arguments*. C'est le développeur qui exécute et renvoie le résultat.

Ce notebook couvre :
- Définir un tool et le passer au modèle
- Cycle complet : appel, exécution, résultat, réponse finale
- Parallel tool calls, quand le modèle appelle plusieurs tools en un échange
- Error handling, signaler un échec au modèle

## Stack

- **OpenAI SDK** (`openai`), client Python officiel
- **Pydantic** , validation des arguments des tools
- **LiteLLM** , approche provider-agnostic en fin de notebook

## What's next

- `02-augmentation/rag` , combiner tools et retrieval
- `02-augmentation/mcp-client` , tools via le protocole MCP
- `03-agentique` , orchestrer plusieurs tools dans une boucle agentique

## Setup

**En local**
1. Copier `.env.example` en `.env` à la racine du repo
2. Renseigner `OPENAI_API_KEY`
3. Lancer le notebook avec ton environnement uv ou jupyter habituel

**Sur Google Colab**
1. Ouvrir les Secrets (icône 🔑 dans le panneau gauche)
2. Ajouter un secret `OPENAI_API_KEY` avec ta clé
3. Activer l'accès au secret pour ce notebook

In [ ]:
%pip install openai pydantic litellm python-dotenv -q

In [ ]:
import os

try:
    from dotenv import load_dotenv
    load_dotenv(override=True)
except ImportError:
    pass  # Colab, variables disponibles via les secrets

assert os.getenv("OPENAI_API_KEY"), (
    "OPENAI_API_KEY manquante, voir .env.example (local) ou Secrets (Colab)"
)

In [ ]:
from openai import OpenAI

client = OpenAI()

## Définir un tool

Un tool est composé de trois éléments :
- **name** , identifiant appelé par le modèle
- **description** , ce que fait le tool, lu par le modèle pour décider de l'appeler
- **parameters** , schéma JSON des arguments attendus

La description est critique : c'est elle qui guide le modèle dans le choix du bon tool.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for current information on a given query. Use this when the user asks about recent events or facts that may have changed.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query",
                    }
                },
                "required": ["query"],
            },
        },
    }
]

In [ ]:
import json
from typing import Any

def web_search(query: str) -> dict[str, Any]:
    """Mock web search, retourne un résultat en dur."""
    return {
        "query": query,
        "results": [
            {
                "title": "France at the 2026 Winter Olympics",
                "url": "https://en.wikipedia.org/wiki/France_at_the_2026_Winter_Olympics",
                "content": "France won 3 gold medals, 2 silver medals and 4 bronze medals at the 2026 Winter Olympics.",
            }
        ],
    }

## Cycle complet

Le cycle tool use se déroule en quatre étapes :

1. **Appel initial** , on envoie le prompt avec les tools disponibles
2. **Tool call** , le modèle répond avec le tool à appeler et ses arguments
3. **Exécution** , on exécute le tool et on renvoie le résultat au modèle
4. **Réponse finale** , le modèle intègre le résultat et répond à l'utilisateur

1. Appel initial

In [ ]:
messages = [
    {"role": "user", "content": "How many medals did France win at the 2026 Winter Olympics?"}
]

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    tools=tools,
)

print("Finish reason:", response.choices[0].finish_reason)
print("Response:", response.choices[0].message)

2. Inspecter le tool call

In [ ]:
tool_call = response.choices[0].message.tool_calls[0]

print("Tool appelé:", tool_call.function.name)
print("Arguments:", tool_call.function.arguments)

# Les arguments sont du JSON sérialisé
args = json.loads(tool_call.function.arguments)
print("Arguments parsés:", args)

3. Exécution du tool et renvoi du résultat

In [ ]:
# Exécution du tool
tool_result = web_search(**args)
print("Résultat du tool:", tool_result)

# On ajoute le message assistant (avec le tool call) et le résultat à l'historique
messages.append(response.choices[0].message)
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(tool_result),
})

4. Réponse finale du LLM

In [ ]:
final_response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    tools=tools,
)

print("Réponse finale:", final_response.choices[0].message.content)

## Parallel tool calls

Le modèle peut décider d'appeler plusieurs tools en un seul échange quand la requête le justifie. On définit deux tools, et on laisse le modèle choisir lesquels appeler.

Le modèle retourne une liste de `tool_calls`, on exécute chacun et on renvoie tous les résultats avant la réponse finale.

1. Définir 2 tools

In [ ]:
tools_parallel = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for current information on a given query.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query"}
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "The city name"}
                },
                "required": ["city"],
            },
        },
    },
]

def get_weather(city: str) -> dict[str, Any]:
    """Mock weather, retourne un résultat en dur."""
    return {
        "city": city,
        "temperature": "12°C",
        "condition": "Partly cloudy",
    }

2. Appel initial parallèle

In [ ]:
messages_parallel = [
    {"role": "user", "content": "What's the weather in Milan and how many medals did France win at the 2026 Winter Olympics?"}
]

response_parallel = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages_parallel,
    tools=tools_parallel,
)

print("Finish reason:", response_parallel.choices[0].finish_reason)
print("Nombre de tool calls:", len(response_parallel.choices[0].message.tool_calls))

for tc in response_parallel.choices[0].message.tool_calls:
    print(f"  Tool: {tc.function.name}, Arguments: {tc.function.arguments}")

3. Exécution parallèle

In [ ]:
tool_registry = {
    "web_search": web_search,
    "get_weather": get_weather,
}

messages_parallel.append(response_parallel.choices[0].message)

for tc in response_parallel.choices[0].message.tool_calls:
    args = json.loads(tc.function.arguments)
    result = tool_registry[tc.function.name](**args)
    print(f"Résultat {tc.function.name}:", result)

    messages_parallel.append({
        "role": "tool",
        "tool_call_id": tc.id,
        "content": json.dumps(result),
    })

3. Réponse finale du LLM

In [ ]:
final_response_parallel = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages_parallel,
    tools=tools_parallel,
)

print("Réponse finale:", final_response_parallel.choices[0].message.content)

## Error handling

Si un tool échoue, on ne cache pas l'erreur. On la renvoie au modèle via le message `tool` pour qu'il puisse adapter sa réponse.

Le modèle est capable d'interpréter une erreur et de répondre à l'utilisateur de façon cohérente plutôt que de halluciner un résultat.

In [ ]:
def web_search_failing(query: str) -> dict[str, Any]:
    """Mock web search qui simule une erreur."""
    raise ConnectionError("Search service unavailable")

messages_error = [
    {"role": "user", "content": "How many medals did France win at the 2026 Winter Olympics?"}
]

response_error = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages_error,
    tools=tools,
)

messages_error.append(response_error.choices[0].message)
tc = response_error.choices[0].message.tool_calls[0]
args = json.loads(tc.function.arguments)

try:
    result = web_search_failing(**args)
    content = json.dumps(result)
except Exception as e:
    # On renvoie l'erreur au modèle plutôt que de crasher
    content = json.dumps({"error": str(e)})
    print("Erreur capturée:", content)

messages_error.append({
    "role": "tool",
    "tool_call_id": tc.id,
    "content": content,
})

## Vers une approche provider-agnostic avec LiteLLM

LiteLLM supporte le function calling avec la même interface que l'OpenAI SDK. La définition des tools et le cycle complet restent identiques, seul le provider change.

In [ ]:
# 👇 Changer ici pour switcher de provider
MODEL = "openai/gpt-4.1-mini"
# MODEL = "anthropic/claude-haiku-4-5"

In [ ]:
from litellm import completion

messages_litellm = [
    {"role": "user", "content": "How many medals did France win at the 2026 Winter Olympics?"}
]

response_litellm = completion(
    model=MODEL,
    messages=messages_litellm,
    tools=tools,
)

tc = response_litellm.choices[0].message.tool_calls[0]
args = json.loads(tc.function.arguments)
result = web_search(**args)

print("Tool appelé:", tc.function.name)
print("Résultat:", result)

messages_litellm.append(response_litellm.choices[0].message)
messages_litellm.append({
    "role": "tool",
    "tool_call_id": tc.id,
    "content": json.dumps(result),
})

final_litellm = completion(
    model=MODEL,
    messages=messages_litellm,
    tools=tools,
)

print("Réponse finale:", final_litellm.choices[0].message.content)